In [66]:
from enum import Enum
from typing import List
import pandas as pd, pandas

class SRActionType(Enum):
    ONE_OPTIONAL = 1
    ONE_MANDATORY = 2
    MANY_OPTIONAL = 3
    MANY_MANDATORY = 4

class ScanResult:
    def __init__(self, row=0, col=0, message="", action_type=SRActionType.ONE_OPTIONAL, actions=None):
        self.row = row
        self.col = col
        self.message = message
        self.action_type = action_type
        self.actions = actions if actions is not None else []

    def add_action(self, action):
        if isinstance(action, ScanResultAction):
            self.actions.append(action)
        else:
            raise TypeError("Only ScanResultAction instances can be added to actions.")
    
    def __repr__(self):
        return f"ScanResult(row={self.row}, col={self.col}, message='{self.message}', action_type={self.action_type}, actions={self.actions})"

class ScanResultAction:
    def __init__(self, title="Title", description="Description", cleaner="", cleaner_id=None, activate=False, data=None):
        self.title = title
        self.description = description
        self.cleaner = cleaner
        self.cleaner_id = cleaner_id
        self.activate = activate
        self.data = data
    
    def __repr__(self):
        return f"ScanResultAction(title='{self.title}', description='{self.description}', cleaner='{self.cleaner}', cleaner_id={self.cleaner_id}, activate={self.activate}, data={self.data})"

In [67]:
import pandas as pd

data = {
    'Name': ['Alice', 'Bob', 'Charlie', 'David', 'Eve', 'John'],
    'Age': [24, 27, 22, 32, 29, 25],
    'City': ['New York', 'Los Angeles', 'California', 'Houston', 'Phoenix', 'Chicago'],
    'Score': [85.5, 90.3, 78.6, 92.1, 88.7, 92.3]
}

df = pd.DataFrame(data)

df

,Name,Age,City,Score
0,Alice,24,New York,85.5
1,Bob,27,Los Angeles,90.3
2,Charlie,22,California,78.6
3,David,32,Houston,92.1
4,Eve,29,Phoenix,88.7
5,John,25,Chicago,92.3


In [68]:
def scan_df_for_chicago(df: pd.DataFrame) -> List[ScanResult]:
    scan_results = []

    for row_index, row in df.iterrows():
        for col_index, col in enumerate(df.columns):
            if str(row[col]).lower() == 'chicago':
                scan_results.append(
                    ScanResult(
                        row=row_index,
                        col=col_index,
                        message=f"Found 'Chicago' in row {row_index+1}, col {col_index+1}",
                        action_type=SRActionType.ONE_MANDATORY,
                        actions=[
                            ScanResultAction(
                                title="Handle Chicago",
                                description="Handle the 'Chicago' value",
                                cleaner="handle_chicago",
                                activate=True,
                            )
                        ]
                    )
                )

    return scan_results

In [69]:
scan_results = scan_df_for_chicago(df)
print(scan_results)

[ScanResult(row=5, col=2, message='Found 'Chicago' in row 6, col 3', action_type=SRActionType.ONE_MANDATORY, actions=[ScanResultAction(title='Handle Chicago', description='Handle the 'Chicago' value', cleaner='handle_chicago', cleaner_id=None, activate=True, data=None)])]


In [70]:
# Cleaner function
# Must follow function format
def handle_chicago(scan_result: ScanResult, df: pandas.DataFrame) -> pandas.DataFrame:
    for action in scan_result.actions:
        if isinstance(action, ScanResultAction):
            df.iat[scan_result.row, scan_result.col] = 'CLEANED'
    return df

In [71]:
scan_results = scan_df_for_chicago(df)

for scan_result in scan_results:
    for action in scan_result.actions:
        if action.activate:
            df = handle_chicago(scan_result, df)

df

,Name,Age,City,Score
0,Alice,24,New York,85.5
1,Bob,27,Los Angeles,90.3
2,Charlie,22,California,78.6
3,David,32,Houston,92.1
4,Eve,29,Phoenix,88.7
5,John,25,CLEANED,92.3
